# Happy / Sad CNN — Stage 2

Reusable dataset/model code lives in `src/`.

This notebook remains responsible for:
- configuration and paths,
- creating the model object,
- compiling,
- training,
- visualising,
- evaluating,
- saving the trained model.


In [ ]:
import sys
print(sys.executable)


In [ ]:
from pathlib import Path

import tensorflow as tf
from matplotlib import pyplot as plt

from image_classifier.dataset import (
    remove_invalid_images,
    load_dataset,
    scale_dataset,
    split_dataset,
)
from image_classifier.model import build_model


In [ ]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("MODELS_DIR:", MODELS_DIR)


## Optional cleaning step


In [ ]:
removed = remove_invalid_images(DATA_DIR)
print(f"Removed {len(removed)} invalid image(s).")


## Load → scale → split


In [ ]:
data = load_dataset(DATA_DIR)
data = scale_dataset(data)

train, val, test = split_dataset(data)

print("Train batches:", len(train))
print("Validation batches:", len(val))
print("Test batches:", len(test))


## Inspect one batch


In [ ]:
batch = train.as_numpy_iterator().next()

print("Images:", batch[0].shape)
print("Labels:", batch[1].shape)
print("Pixel min:", batch[0].min())
print("Pixel max:", batch[0].max())


In [ ]:
fig, axes = plt.subplots(ncols=4, figsize=(16, 4))

for idx, image in enumerate(batch[0][:4]):
    axes[idx].imshow(image)
    axes[idx].set_title(f"Label: {batch[1][idx]}")
    axes[idx].axis("off")

plt.show()


## Build the model in Jupyter

`build_model()` defines the reusable architecture in `src/model/cnn.py`.
Calling it here creates the actual Keras model object in this notebook session.


In [ ]:
model = build_model()
model.summary()


## Compile in Jupyter


In [ ]:
model.compile(
    optimizer="adam",
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=["accuracy"],
)


## Train in Jupyter


In [ ]:
history = model.fit(
    train,
    epochs=20,
    validation_data=val,
)


## Plot loss


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


## Plot accuracy


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="accuracy")
plt.plot(history.history["val_accuracy"], label="val_accuracy")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


## Evaluate on the held-out test set


In [ ]:
precision = tf.keras.metrics.Precision()
recall = tf.keras.metrics.Recall()
accuracy = tf.keras.metrics.BinaryAccuracy()

for x_batch, y_true in test:
    y_pred = model.predict(x_batch, verbose=0)

    precision.update_state(y_true, y_pred)
    recall.update_state(y_true, y_pred)
    accuracy.update_state(y_true, y_pred)

print("Precision:", float(precision.result()))
print("Recall:", float(recall.result()))
print("Accuracy:", float(accuracy.result()))


## Save the trained model


In [ ]:
MODELS_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODELS_DIR / "image_classifier.keras"
model.save(MODEL_PATH)

print("Saved to:", MODEL_PATH)
